# 03 · Silver → Gold

Conceito praticado: **Star Schema** em pequena escala
- `dim_customer`, `dim_product`, `dim_date` (dimensões, com chave substituta / surrogate key)
- `fact_order_items` (fato, na granularidade de item de pedido)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. dim_date
Gerada a partir das datas de compra distintas em `silver.orders`.

In [0]:
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
df_orders_silver = spark.table(f"{silver_schema}.orders")

df_dim_date = (
    df_orders_silver
    .select(F.to_date("data_compra_pedido").alias("full_date"))
    .filter(F.col("full_date").isNotNull())
    .distinct()
    .withColumn("date_sk", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .select("date_sk", "full_date", "year", "month", "day", "quarter", "month_name", "day_of_week")
)

df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_date")

print(f"dim_date: {df_dim_date.count()} datas distintas")

dim_date: 634 datas distintas


## 2. dim_customer

In [0]:
df_customers_silver = spark.table(f"{silver_schema}.customers")

df_dim_customer = (
    df_customers_silver
    .withColumn("customer_sk", F.row_number().over(Window.orderBy("id_consumidor")))
    .select("customer_sk", "id_consumidor", "id_cliente_unico",
            "cidade_cliente", "estado_cliente", "prefixo_cep_cliente")
)

df_dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_customer")

print(f"dim_customer: {df_dim_customer.count()} clientes")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customer: 99441 clientes


## 3. dim_product

In [0]:
df_products_silver = spark.table(f"{silver_schema}.products")

df_dim_product = (
    df_products_silver
    .withColumn("product_sk", F.row_number().over(Window.orderBy("id_produto")))
    .select("product_sk", "id_produto", "categoria_produto",
            "categoria_produto_ingles", "peso_produto_gramas",
            "comprimento_centimetros", "altura_centimetros", "largura_centimetros")
)

df_dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_product")

print(f"dim_product: {df_dim_product.count()} produtos")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_product: 32951 produtos


## 4. fact_order_items
Granularidade: **um item de pedido por linha**.
Pagamentos são agregados por `order_id` (um pedido pode ter mais de um pagamento) e
associados a todos os itens daquele pedido — simplificação didática para a aula.

In [0]:
df_items_silver = spark.table(f"{silver_schema}.order_items")

df_orders_ctx = (
    spark.table(f"{silver_schema}.orders")
    .select("id_pedido", "id_cliente", "status_pedido",
            F.to_date("data_compra_pedido").alias("order_date"))
)

df_payments_agg = (
    spark.table(f"{silver_schema}.order_payments")
    .groupBy("id_pedido")
    .agg(
        F.sum("valor_pagamento").alias("total_payment_value"),
        F.first("forma_pagamento").alias("main_payment_type"),
        F.sum("parcelas").alias("total_installments"),
    )
)

df_dim_customer_lk = spark.table(f"{gold_schema}.dim_customer").select("customer_sk", "id_consumidor")
df_dim_product_lk = spark.table(f"{gold_schema}.dim_product").select("product_sk", "id_produto")
df_dim_date_lk = spark.table(f"{gold_schema}.dim_date").select("date_sk", "full_date")

df_fact_order_items = (
    df_items_silver
    .join(df_orders_ctx, on="id_pedido", how="inner")
    .join(df_payments_agg, on="id_pedido", how="left")
    .join(df_dim_customer_lk, F.col("id_cliente") == F.col("id_consumidor"), how="left")
    .join(df_dim_product_lk, on="id_produto", how="left")
    .join(df_dim_date_lk, df_orders_ctx["order_date"] == df_dim_date_lk["full_date"], how="left")
    .select(
        F.monotonically_increasing_id().alias("fact_sk"),
        "id_pedido",
        "id_item",
        "customer_sk",
        "product_sk",
        "date_sk",
        "status_pedido",
        F.col("preco").alias("item_price"),
        F.col("preco_frete").alias("item_freight_value"),
        "total_payment_value",
        "main_payment_type",
        "total_installments",
    )
)

df_fact_order_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_order_items")

print(f"fact_order_items: {df_fact_order_items.count()} linhas")

fact_order_items: 112650 linhas


## 5. Testando o Star Schema
Receita por estado do cliente e mês da compra — prova de que fato e dimensões se conectam.

In [0]:
df_resultado = (
    spark.table(f"{gold_schema}.fact_order_items").alias("f")
    .join(spark.table(f"{gold_schema}.dim_customer").alias("c"), "customer_sk")
    .join(spark.table(f"{gold_schema}.dim_date").alias("d"), "date_sk")
    .groupBy("c.estado_cliente", "d.year", "d.month")
    .agg(
        F.round(F.sum("f.item_price"), 2).alias("receita_produtos"),
        F.round(F.sum("f.item_freight_value"), 2).alias("receita_frete"),
        F.count("*").alias("qtd_itens"),
    )
    .orderBy(F.desc("receita_produtos"))
)

display(df_resultado)

estado_cliente,year,month,receita_produtos,receita_frete,qtd_itens
SP,2018,5,436596.30,54781.04,3685
SP,2018,4,398822.42,53175.85,3551
SP,2018,8,396613.49,56896.32,3652
SP,2018,3,385708.78,53091.39,3470
SP,2018,1,375404.50,50596.88,3434
SP,2017,11,360251.89,51391.00,3437
SP,2018,6,343828.57,52055.02,3235
SP,2018,7,326319.19,51232.43,3162
SP,2018,2,312318.98,43954.77,3089
SP,2017,12,280740.14,38990.06,2646


## Recapitulando a arquitetura medalhão construída

| Camada   | O que fizemos |
|----------|----------------|
| Landing  | 6 arquivos `.csv` brutos em um Volume |
| Bronze   | 1 lote por arquivo → Delta, sem transformação de negócio, com metadados de auditoria |
| Silver   | Deduplicação, tratamento de nulos, conversão de tipos, testes de qualidade logados |
| Gold     | Star Schema: `dim_customer`, `dim_product`, `dim_date`, `fact_order_items` |

Todas as tabelas são Delta Lake: transações ACID, schema enforcement e Time Travel em todas as camadas.